In [1]:
!pip -q install openai backoff --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.9/362.9 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.9 MB/s eta 0:00:00


In [2]:
# Download der Bilder
!unzip -q ../data/insta-story-images.zip -d ./images

In [24]:
import pandas as pd

df = pd.read_csv('../data/2024-07-03-Faces-in-Stories-unfiltered.csv')

In [25]:
df.head()

,Unnamed: 0,face_uuid,account_name,Ground Truth,filename,match_file,match_score,model,distance_metric,number_of_hits,Model
0,0,0b8b3aa1-e602-4d45-a088-e7f3b57e2306,armin_laschet,Armin Laschet,armin_laschet_26-09-2021_00:00_video2.jpeg,c9b28a08-82fc-4524-9287-70b4dbc96c5b.png,0.798309,Facenet512,euclidean_l2,18.0,Armin Laschet
1,1,f2cc0ff3-6696-4d59-a762-2073cae25025,armin_laschet,Unknown,armin_laschet_26-09-2021_00:00_video2.jpeg,NaN,NaN,NaN,NaN,NaN,Unknown
2,2,d5890112-c278-4c09-ac0c-13e4ecef8790,armin_laschet,Unknown,armin_laschet_26-09-2021_00:00_video2.jpeg,NaN,NaN,NaN,NaN,NaN,Unknown
3,3,842af872-d788-4629-aa74-b031cea55fe2,armin_laschet,Unknown,armin_laschet_14-09-2021_00:00_video3.jpeg,NaN,NaN,NaN,NaN,NaN,Unknown
4,4,523fccf4-70e1-437e-bb1d-c906fd0d4648,spdde,Olaf Scholz,spdde_26-09-2021_00:00_video1.jpeg,cf207ba3-2a6b-42b2-b0c6-b0fb9e7532f0.png,0.754718,Facenet512,euclidean_l2,19.0,Olaf Scholz


## Counting People

In [26]:
df_filtered = df[df['Ground Truth'] != 'Unknown']

In [ ]:
# Prompt V1
prompt = """
Imagine you were a social science researcher, conducting an analysis of the 2021 German Federal election campaign. The image shows the front-runner(s): {NAME}. We're interested in measuring individualization of candidates in the election campaign.  How many people are in the focus of the image? Just the candidate, a small group of politicians, or a large group? Count the number of clearly visible individuals.
Select the right choice: 'GPT Count': ["0", "1", "2", "3+"]

Respond in valid JSON only.
"""

# Additionally: Is there a crowd of spectators visible? 'GPT Crowd': ["True", "False"]


In [ ]:
# Prompt V2
prompt = """
Imagine you were a social science researcher, conducting an analysis of the 2021 German Federal election campaign. The image shows the front-runner(s): {NAME}. We're interested in measuring individualization of candidates in the election campaign.  How many people are in the focus of the image? Just the candidate, a group of two people, or more than two people? Count the number of clearly visible individuals.
Select the right choice: 'GPT Count': ["0", "1", "2", "3+"]
Additionally: Is there a crowd of spectators visible? 'GPT Crowd': ["True", "False"]

Respond in valid JSON only.
"""

# Additionally: Is there a crowd of spectators visible? 'GPT Crowd': ["True", "False"]


In [27]:
# Prompt V3
prompt = """
Imagine you are a social science researcher, conducting an analysis of the 2021 German Federal election campaign. The image shows the front-runner: {NAME}. We're interested in measuring the individualization of candidates in the election campaign.

Assess the image and provide the following information:

1. How many people are in the focus of the image? Select the right choice: 'GPT Count': ["0", "1", "2", "3+"]
2. Is there a crowd of spectators visible? Select the right choice: 'GPT Crowd': ["True", "False"]

Respond in valid JSON only.
"""

In [6]:
# Prompt V3
prompt = """
Imagine you are a social science researcher, conducting an analysis of the 2021 German Federal election campaign. The image shows the front-runner: {NAME}. We're interested in measuring the individualization of candidates in the election campaign.

Assess the image and provide the following information:

1. How many people are in the focus of the image? Select the right choice: 'GPT Count': ["0", "1", "2", "3+"]
2. Is there a crowd of spectators visible? Select the right choice: 'GPT Crowd': ["True", "False"]

Respond in valid JSON only.
"""

In [ ]:
# Prompt V4
prompt = """
Imagine you are a social science researcher, conducting an analysis of the 2021 German Federal election campaign. The image shows the front-runner: {NAME}. We're interested in measuring the individualization of candidates in the election campaign.

Assess the image and provide the following information:

1. Is the front-runner pictured by himself / herself? Select the right choiche 'GPT Self': ["True", "False"]
2. Is there a crowd of spectators visible? Select the right choice: 'GPT Crowd': ["True", "False"]

Respond in valid JSON only.
"""

In [ ]:
# Prompt V5
prompt = """
Imagine you are a social science researcher conducting an analysis of the 2021 German Federal election campaign. The image shows the front-runner: {NAME}. We are interested in measuring the individualization of candidates in the election campaign, specifically by assessing whether the front-runner is the only person in focus in the image.

Please evaluate the image and provide the following information in valid JSON format:

1. Is only {NAME} in focus in the image? (Choices: "True", "False")
2. Is there a crowd of spectators visible in the image? (Choices: "True", "False")

Example response:
```json
{
  "GPT Self": "True",
  "GPT Crowd": "False"
}
```
"""

In [ ]:
# Prompt V6
prompt = """
Imagine you are a social science researcher conducting an analysis of the 2021 German Federal election campaign. Based on human annotations we know that the image shows the front-runner {NAME}. We are interested in measuring the individualization of candidates in the election campaign, specifically by assessing whether the front-runner is the only person in focus in the image.

Please evaluate the image and provide the following information in valid JSON format:
1. Is only the front-runner in focus in the image? (Choices: "True", "False", "No Person")
2. Is there a crowd of spectators visible in the image? (Choices: "True", "False")

## Example response:
{
 "GPT Self": "True" / "False" / "No One",
 "GPT Crowd": "True" / "False"
}
"""

In [28]:
df_filtered.head()

,Unnamed: 0,face_uuid,account_name,Ground Truth,filename,match_file,match_score,model,distance_metric,number_of_hits,Model
0,0,0b8b3aa1-e602-4d45-a088-e7f3b57e2306,armin_laschet,Armin Laschet,armin_laschet_26-09-2021_00:00_video2.jpeg,c9b28a08-82fc-4524-9287-70b4dbc96c5b.png,0.798309,Facenet512,euclidean_l2,18.0,Armin Laschet
4,4,523fccf4-70e1-437e-bb1d-c906fd0d4648,spdde,Olaf Scholz,spdde_26-09-2021_00:00_video1.jpeg,cf207ba3-2a6b-42b2-b0c6-b0fb9e7532f0.png,0.754718,Facenet512,euclidean_l2,19.0,Olaf Scholz
7,7,b8205fa2-9302-4ba6-82ff-00587434e0f9,armin_laschet,Armin Laschet,armin_laschet_20-09-2021_00:00_video6.jpeg,1e103923-022b-4d3f-a32e-e9c848a00e70.png,0.631817,Facenet512,euclidean_l2,20.0,Armin Laschet
10,39,a44eed36-cfa8-4187-b3f4-af8ccf7c5241,christianlindner,Christian Lindner,christianlindner_25-09-2021_00:00_video0.jpeg,b2b81709-caad-45cc-b632-78901d5ab389.png,0.763754,Facenet512,euclidean_l2,7.0,Christian Lindner
12,41,6d7abb70-94a7-4928-a327-f6e77a32bb31,abaerbock,Annalena Baerbock,abaerbock_18-09-2021_00:00_image6.jpeg,NaN,NaN,NaN,NaN,NaN,Unknown


In [29]:
df_filtered = df_filtered.groupby('filename').agg({
    'Ground Truth': lambda x: ', '.join(set(x)),
    'account_name': 'first',
    }).reset_index()

In [30]:
df_filtered.head()

,filename,Ground Truth,account_name
0,abaerbock_13-09-2021_00:00_image0.jpeg,Annalena Baerbock,abaerbock
1,abaerbock_13-09-2021_00:00_image1.jpeg,Annalena Baerbock,abaerbock
2,abaerbock_13-09-2021_00:00_image2.jpeg,Annalena Baerbock,abaerbock
3,abaerbock_13-09-2021_00:00_image3.jpeg,Annalena Baerbock,abaerbock
4,abaerbock_13-09-2021_00:00_image4.jpeg,Annalena Baerbock,abaerbock


In [31]:
len(df_filtered)

616

In [32]:
import pandas as pd
import json
import re
from pandas import json_normalize

def flatten_json(y):
    out = {}

    def flatten(x, name=''):
        if type(x) is dict:
            for a in x:
                flatten(x[a], name + a + '_')
        elif type(x) is list:
            i = 0
            for a in x:
                flatten(a, name + str(i) + '_')
                i += 1
        else:
            out[name[:-1]] = x

    flatten(y)
    return out

def parse_response(response, identifier):
    try:
        if isinstance(response, str):
            response = json.loads(response)
        response = flatten_json(response)
        response['Image'] = identifier
        return response
    except json.JSONDecodeError:
        match = re.search(r'```json\n([\s\S]+)\n```', response)
        if match:
            try:
                json_data = json.loads(match.group(1))
                json_data = flatten_json(json_data)
                json_data['Image'] = identifier
                return json_data
            except json.JSONDecodeError:
                pass
        return {'Image': identifier, 'error': response}

---
Kleiner Einwurf: Die Evaluation mit GPT-4o war sehr schlecht. Deshalb nochmal Test mit dem GPT-4 Modell.

In [12]:
df = pd.read_csv('../data/2024-07-04-Annotated-Images-Person-Count.csv')

In [ ]:
df.head()

,Unnamed: 0,Image,7107,10475,16195,Majority Decision,Gold,GPT-4,GPT-4o,GPT-4-Neu
0,0,abaerbock_26-09-2021_00:00_video0.jpeg,1,1,1,1,1,1,1,1
1,2,armin_laschet_21-09-2021_00:00_video1.jpeg,1,1,1,1,1,1,1,1
2,3,abaerbock_25-09-2021_00:00_video0.jpeg,3+,3+,3+,3+,3+,3+,3+,3+
3,4,olafscholz_24-09-2021_00:00_image2.jpeg,2,2,2,2,2,2,2,2
4,5,die_gruenen_13-09-2021_00:00_image2.jpeg,3+,3+,3+,3+,3+,3+,3+,3+


In [ ]:
df_filtered = df_filtered[df_filtered['filename'].isin(df['Image'])]

---

In [33]:
old_df = pd.read_csv('../data/2024-07-04-Stories-Count-Annotation-Not-Sampled-GPT4o-V3-TEST.csv')

In [34]:
old_df.head()

,Unnamed: 0,GPT Count,GPT Crowd,Image
0,0,2,False,abaerbock_13-09-2021_00:00_image1.jpeg
1,1,2,False,abaerbock_13-09-2021_00:00_video1.jpeg
2,2,1,False,abaerbock_14-09-2021_00:00_image11.jpeg
3,3,1,True,abaerbock_14-09-2021_00:00_image4.jpeg
4,4,1,True,abaerbock_14-09-2021_00:00_video3.jpeg


In [36]:
df_filtered = df_filtered[~df_filtered['filename'].isin(old_df['Image'])]

In [37]:
len(df_filtered)

423

In [38]:
import base64
import requests
from tqdm.notebook import tqdm
import time
import openai
from openai import OpenAI
import os

api_key = os.environ.get("OPENAI_API_KEY")
api_key = os.environ.get("OPENAI_API_KEY")


# OpenAI API Key
client = OpenAI(
    api_key=api_key
)

prompt_cost = 0.01 / 1000
completion_cost = 0.03 / 1000

total_cost = 0.0

# Function to encode the image
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

import backoff
@backoff.on_exception(backoff.expo, (openai.RateLimitError, openai.APIError))
def run_request(prompt, base64_image):
  messages = [{
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}",
                        }
                    }
                ]
            }]


  return client.chat.completions.create(
                    model="gpt-4o-2024-05-13",
                    temperature=0,
                    messages=messages,
                    max_tokens=600)

responses = []
data = []

for index, row in tqdm(df_filtered.iterrows(), total=len(df_filtered)):
    image = row['filename']

    # Path to your image
    image_path = f"/content/images/upload_s3/{image}"

    # Getting the base64 string
    base64_image = encode_image(image_path)

    tmp_prompt = prompt.replace('{NAME}', row['Ground Truth'])
    response = run_request(tmp_prompt, base64_image)

    # Extract the response content
    r = response.choices[0].message.content

    # Adding prompt cost
    current_prompt_cost = response.usage.prompt_tokens * prompt_cost
    current_completion_cost = response.usage.completion_tokens * completion_cost

    current_cost = current_prompt_cost + current_completion_cost
    total_cost += current_cost


    processed_data = parse_response(r, image)
    data.append(processed_data)

print(f"Total cost ${total_cost}")

  0%|          | 0/423 [00:00<?, ?it/s]

Total cost $3.98817999999999


In [39]:
response_df = pd.DataFrame(data)

In [40]:
response_df.head()

,GPT Count,GPT Crowd,Image
0,1,False,abaerbock_13-09-2021_00:00_image0.jpeg
1,1,False,abaerbock_13-09-2021_00:00_image2.jpeg
2,1,False,abaerbock_13-09-2021_00:00_image3.jpeg
3,1,False,abaerbock_13-09-2021_00:00_image4.jpeg
4,1,False,abaerbock_13-09-2021_00:00_image5.jpeg


In [42]:
response_df = pd.concat([response_df, old_df], ignore_index=True)

In [45]:
response_df.to_csv('../data/2024-07-04-Stories-Count-Annotation-Not-Sampled-GPT4o-V3-TEST.csv', index=False)

In [43]:
response_df

,GPT Count,GPT Crowd,Image,Unnamed: 0
0,1,False,abaerbock_13-09-2021_00:00_image0.jpeg,NaN
1,1,False,abaerbock_13-09-2021_00:00_image2.jpeg,NaN
2,1,False,abaerbock_13-09-2021_00:00_image3.jpeg,NaN
3,1,False,abaerbock_13-09-2021_00:00_image4.jpeg,NaN
4,1,False,abaerbock_13-09-2021_00:00_image5.jpeg,NaN
...,...,...,...,...
611,2,False,spdde_21-09-2021_00:00_video0.jpeg,188.0
612,2,False,spdde_26-09-2021_00:00_video10.jpeg,189.0
613,0,False,spdde_26-09-2021_00:00_video11.jpeg,190.0
614,0,False,spdde_26-09-2021_00:00_video12.jpeg,191.0


In [ ]:
response_df['GPT Count'] = response_df.apply(lambda row: row['GPT Count_0'] if pd.isna(row['GPT Count']) else row['GPT Count'], axis=1)

In [ ]:
total_df = pd.merge(response_df, df, on="Image", how="left")

In [ ]:
total_df['GPT Count'] = total_df['GPT Count'].apply(lambda x: x.replace("\"", ""))

In [ ]:
total_df = total_df[["Image", "Human Count", "Face Count", "GPT Count", "People"]]

In [ ]:
total_df.to_csv('../data/2023-12-21-Multiple-Counts-Story-Faces-w-GPT.csv')